# 策略概述

**Z-Score 狀態機**是本研究的**傳統交易基準**：把形成期選出的每一組配對，依「價差偏離幾倍標準差」機械式地進出場——偏離過大時進場（賭它會回歸），回到均值時獲利了結。

所有形成期策略（SSD、DTW、HDBSCAN、Agglomerative）的交易端都用這個引擎；命題 2 的深度學習交易（DRL）也以它為對照。它的門檻是**固定的**，這正是 DRL 想改進之處。


# 策略架構

交易期對每組配對逐日推進，分五個環節；**進出場門檻固定**是它與 DRL 版的關鍵差異。

```{mermaid}
flowchart LR
  A["價差 / Z-Score<br/>形成期統計量重建"] --> B["訊號<br/>偏離進場、回歸出場（固定門檻）"]
  B --> C["部位<br/>風險中性配置"]
  C --> D["風控<br/>比例停損 + 發散停損 + 冷卻"]
  D --> E["期末<br/>未平倉強制結算"]
```

| 環節 | 本引擎作法 | 用途 |
| :--- | :--- | :--- |
| 價差重建 | 以形成期凍結統計量在標準化空間重建 | 與形成期一致，避免座標偏移 |
| 訊號 | 偏離 2 倍標準差進場、回歸均值出場 | **固定門檻**（DRL 版改為學習挑選） |
| 部位配置 | 依對沖比例分配兩腳資金（市場中性） | 對沖市場風險 |
| 風控 | 比例停損、價差發散停損、方向冷卻 | 控制單筆與續擴損失 |
| 期末 | 未平倉部位強制結算 | 每期獨立結算 |


# 參考文獻與引用對應


## 文獻 1：Gatev, Goetzmann & Rouwenhorst (2006)

> Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs trading: Performance of a relative value arbitrage rule. *Review of Financial Studies, 19*(3), 797–827.

**參考部分**：

- 第 3 節交易規則：價差偏離超過**形成期 2 倍標準差**時開倉（做空高估腳、做多低估腳），
  價差**收斂交叉**時平倉，交易期結束時強制平倉
- 形成期統計量在交易期**保持不變**的靜態設計
- 等金額（市場中性）對沖建倉

**為何參考**：

- 本引擎的核心狀態機即此交易規則的實作：`entry_z = 2.0` 對應 2 倍標準差開倉、
  `exit_z = 0.0` 對應收斂平倉、`PERIOD_END_EXIT` 對應期末強制結算
- `zscore_window = 0`（靜態模式）對應其形成期統計量凍結設計



## 文獻 2：Do & Faff (2012)

> Do, B., & Faff, R. (2012). Are pairs trading profits robust to trading costs? *Journal of Financial Research, 35*(2), 261–287.

**參考部分**：

- 配對交易獲利對**交易成本（佣金、市場衝擊、買賣價差）**的敏感性實證：
  未計成本的距離法獲利在納入實際成本後大幅縮水

**為何參考**：

- 本引擎**每一筆進出場都扣除摩擦成本**（`fee_rate = 0.0029` + `slippage_rate = 0`，
  往返合計 0.58%（0.29%×2），以兩腳名目金額計算）的設計依據——
  避免回測結果高估未計成本的獲利



## 文獻 3：Krauss, Do & Huck (2016)

> Krauss, C., Do, X. A., & Huck, N. (2016). The profitability of pairs trading strategies: Distance, cointegration and copula methods. *European Journal of Operational Research*.

**參考部分**：

- 停損機制對配對交易風險的控制作用：截斷未收斂配對的損失尾部，但需權衡觸發頻率
- 交易成本必須納入回測假設

**為何參考**：

- 比例停損（`stop_loss_pct` 網格 [0, 5%, 15%]）與 Z 發散停損（`dynamic_stop_z`）
  兩道風控的設計依據；網格搜尋讓停損強度成為可比較的實驗變因



# 各階段行為

引擎對每個交易期（126 日）的每組配對獨立執行以下流程；
滾動步長 21 日使同一時刻最多有 6 個交易期重疊，每期每配對配置獨立資金 `capital_per_pair`。


## 階段 1：價格資料清理

進入模擬前，對整個價格矩陣做一次異常跳動清理：

- 單日漲跌幅**絕對值 > 50%** 的價格點視為資料異常（拆股未調整、錯誤報價），
  以前值遞補（`ffill` 後 `bfill`）
- 兩股取共同交易日交集；有效交易日不足 5 日的配對跳過


## 階段 2：Spread 與 Z-Score 重建（三種座標路徑）

依形成期輸出的欄位自動選擇座標路徑，**與形成期使用完全相同的空間**：

| 路徑 | 判定條件 | Spread 定義 | 對應形成期策略 |
| :---: | :--- | :--- | :--- |
| A | `OLS_Alpha` 存在且未被忽略 | $\ln P_A - \alpha - \beta \ln P_B$ | （現役無；保留供附錄策略使用） |
| B | `Log_Mean/Std` 存在 | $P'_A - \beta P'_B$，$P'_i = \frac{\ln P_i - \mu_i^{form}}{\sigma_i^{form}}$ | **現役全部策略**（HDBSCAN／Agglomerative／K-means／GICS × SSD／DTW／SDP） |
| B1 | `First_Price` 存在 | $\tilde{P}_A - \tilde{P}_B$，$\tilde{P}_i = P_i / P_{i,0}^{form}$ | 附錄 A：Gatev (2006) 原型復刻 |

**Z-Score**（靜態模式 `zscore_window = 0`，形成期統計量全期凍結）：

$$Z_t = \text{clip}\left(\frac{\text{Spread}_t - \mu_\epsilon^{form}}{\max(\sigma_\epsilon^{form},\ \sigma_{min})},\ -10,\ 10\right)$$

- $\sigma_{min} = 10^{-6}$ 防除零；$\pm 10$ 截尾防極端值污染
- 選配：`use_vol_adjust = True` 時分母放大為 $\max(1, \sigma_{20d}/\sigma^{form}) \cdot \sigma^{form}$（現役預設關閉）
- 選配：`zscore_window > 0` 時改用滾動 OLS 重估 $\beta$ 與標準差（現役預設 0，不啟用）


## 階段 3：進場（依據：Gatev et al. 2006）

無持倉時逐日檢查：

| 條件 | 動作 |
| :--- | :--- |
| $Z_t > entry\_z$（= 2.0） | **做空 spread**：空 A、多 B |
| $Z_t < -entry\_z$ | **做多 spread**：多 A、空 B |

**風險中性部位配置**（兩腳資金依對沖比例分配）：

$$W = 1 + |\beta|, \qquad v_A = \frac{C_{pair}}{W}, \qquad v_B = \frac{|\beta| \cdot C_{pair}}{W}$$

$$n_A = \pm\frac{v_A}{P_A}, \qquad n_B = \mp\frac{v_B}{P_B}$$

**進場成本**（立即認列）：

$$\text{EntryFee} = \big(|n_A| P_A + |n_B| P_B\big) \times (\text{fee} + \text{slippage})$$


## 階段 4：持倉評價與出場

持倉期間逐日計算含成本的浮動損益（出場費用以當日價格預估）：

$$\text{TradePnL}_t = \underbrace{n_A (P_{A,t} - P_{A,entry}) + n_B (P_{B,t} - P_{B,entry})}_{\text{原始未實現}} - \text{EntryFee} - \widehat{\text{ExitFee}}_t$$

**出場條件**（依據：Gatev et al. 2006 的收斂平倉）：

| 條件 | 狀態標記 |
| :--- | :--- |
| 空頭持倉且 $Z_t \le exit\_z$（= 0.0） | `EXIT` |
| 多頭持倉且 $Z_t \ge -exit\_z$ | `EXIT` |
| 交易期最後一日仍持倉 | `PERIOD_END_EXIT`（強制結算） |

出場時 `TradePnL` 轉入已實現損益；同一配對出場後可再次進場（等待下一次突破）。


## 階段 5：停損與凍結（依據：Krauss et al. 2016）

持倉期間優先於出場檢查：

| 機制 | 觸發條件 | 行為 |
| :--- | :--- | :--- |
| **比例停損** | $-\text{TradePnL}_t / C_{pair} \ge stop\_loss\_pct$（網格 0／5%／15%；0 = 不啟用） | 立即平倉 |
| **Z 發散停損** | `use_dynamic_stop` 且 $|Z_t| > dynamic\_stop\_z$ | 立即平倉（判定均衡結構破裂） |

**凍結規則**：停損觸發後（`allow_reentry = False`，現役預設），
該配對**本交易期剩餘日全部凍結**（狀態 `STOPPED`，不再進場）——
停損被視為「此配對的形成期均衡關係已失效」的訊號，本期內不再信任該配對。


## 階段 6：選配機制（現役預設關閉）

| 機制 | 參數 | 行為 |
| :--- | :--- | :--- |
| 市場波動政體過濾 | `vol_regime_threshold > 0` | 市場 30 日年化波動率超過閾值時暫停**新開倉**（持倉不受影響） |
| 波動率自適應 Z | `use_vol_adjust = True` | 以 20 日滾動波動放大 Z-Score 分母（見階段 2） |
| 收斂持有模式 | `hold_to_period_end = True` | 進場後不做 Z 回歸出場，持有至期末（停損仍有效） |
| 停損後再進場 | `allow_reentry = True` | 停損不凍結，可再次進場 |

這些機制保留為實驗開關，現役網格皆使用預設值（全部關閉）。


## 階段 7：輸出與損益會計

每配對每日輸出一列交易日誌：

| 欄位 | 內容 |
| :--- | :--- |
| `ZScore` / `Position` | 當日訊號值與持倉方向（−1／0／+1） |
| `Status` | `HOLD_CASH`、`ENTER_LONG_A`、`ENTER_SHORT_A`、`HOLDING`、`EXIT`、`STOP_LOSS_TRIGGERED`、`PERIOD_END_EXIT`、`STOPPED` |
| `Unrealized_PnL` / `Realized_PnL` / `Cumulative_PnL` | 浮動／已實現／累計損益 |
| `Trade_PnL` | 平倉當日的單筆已實現損益（其餘日為 0） |
| `Daily_Delta` | 累計損益的逐日變化（績效統計的基礎數列） |

**會計原則**：`Trade_PnL` 只在平倉日記帳；期間風險以 `Unrealized_PnL` 呈現；
`Daily_Delta` = 當日累計損益 − 前日累計損益，跨配對加總即得組合層級的日損益。


# 參數總表

| 參數 | 值 | 對應環節 | 說明 |
| :--- | :---: | :--- | :--- |
| 交易期長度 | 126 交易日 | 全流程 | 約半年，最多 6 期重疊 |
| 每配對資金 | 10,000 | 部位配置 | 每組配對獨立資金 |
| 進場 / 出場門檻 | 2.0 / 0.0 倍標準差 | 訊號 | 偏離進場、回歸出場（部分策略掃描進場門檻） |
| 統計量模式 | 形成期全期凍結 | 價差重建 | 交易期不重估 |
| 價差截尾 | ±10 | 價差重建 | 避免極端 Z 值 |
| 往返交易成本 | 0.58% | 風控/損益 | 進出場各按兩腳名目額計 |
| 比例停損 | 網格 [0, 5%, 15%] | 風控 | 單筆虧損上限 |
| 發散停損 | 部分策略網格 [0, 3, 4] | 風控 | 價差續擴時的停損水位 |
| 停損後再進場 | 否 | 風控 | 停損後該配對本期凍結 |
